In [1]:
import os
import torch
import pickle
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from lib.model.resNet.resNet import ResNet
from tqdm import tqdm
from configs.test import parser

In [2]:
args = parser.parse_args(['--num_workers', '0', "--checkEpoch", "10"])
print(args)

Namespace(dataPath='./data', num_workers=0, batchSize=64, device='cuda', use_tensorboard=True, tensorboard_dir='./output/tensorboard/', log_interval=10, checkpoint_dir='./output/checkpoints/', checkSession=1, checkEpoch=10, checkPoint=469, res_dir='./output/predictions/')


In [3]:
if args.use_tensorboard:
    log_dir = os.path.join(args.tensorboard_dir, str(args.checkSession))
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
    writer = SummaryWriter(log_dir=log_dir)

In [4]:
def get_fashion_mnist_labels(labels):
    """return text-labels of Fashion-mnist dataset"""
    text_labels = ['t-shirt', 'trouser', 'pullover', 'dress', 'coat', 
                  'sandal', 'shirt', 'sneaker', 'bag', 'ankle boot']
    return [text_labels[int(i)] for i in labels]

# pre-process
# type: PIL -> torch.float32.Tensor
# pixel value in [0, 1]
# resize to (256, 256)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256))])

mnist_test = datasets.FashionMNIST(
    root=args.dataPath, train=False, transform=transform, download=False)

# mnist_train: (tuple0, tuple1, ...)
# tuplei: (image, label)
# image: shape=(C, H, W)
testSize = len(mnist_test)
print("test size: {}".format(testSize))
print("sample 0 image shape: {}".format(mnist_test[0][0].shape))
print("sample 0 label format: {}".format(mnist_test[0][1]))

DataLoaderTest = DataLoader(
    mnist_test, batch_size=args.batchSize, num_workers=args.num_workers)

test size: 10000
sample 0 image shape: torch.Size([1, 256, 256])
sample 0 label format: 9


In [5]:
resNet = ResNet()
resNet.adaptMnist()
# load checkpoint
checkpointName = 'resNet_{}_{}_{}.pth'.format(
    args.checkSession, args.checkEpoch, args.checkPoint)
print(">>> load checkpoint : {}".format(checkpointName))
checkpointPath = os.path.join(
    args.checkpoint_dir, str(args.checkSession), str(checkpointName))
# check checkpoint path exists
if not os.path.exists(checkpointPath):
    raise FileNotFoundError(
        ">>> No checkpoint found at: {}".format(checkpointPath))
checkpoint = torch.load(checkpointPath, map_location=args.device)
resNet.load_state_dict(checkpoint['model_state_dict'])
print(">>> checkpoint loaded")

criterion = torch.nn.CrossEntropyLoss()
resNet.eval()
resNet.to(args.device)

>>> Initializing ResNet model.
>>> conv1 weight shape: (3, 64) - > (1, 64)
>>> fc weight shape: (512, 1000) - > (512, 10)
>>> load checkpoint : resNet_1_10_469.pth


/tmp/ipykernel_38630/2867509553.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpointPath, map_location=args.device)


>>> checkpoint loaded


ResNet(
  (model): ResNet(
    (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_runnin

In [14]:
# set log_interval to checkPoint for test
iters_per_epoch = int((testSize + args.batchSize -1) / args.batchSize)
args.log_interval =iters_per_epoch 
res = torch.zeros(testSize, dtype=torch.int8)
if args.use_tensorboard:
    loss_avg_temp = 0.0
    acc_avg_temp = 0.0
for step, (images, label) in tqdm(enumerate(DataLoaderTest)):
    # move data to device
    images = images.to(args.device)
    label = label.to(args.device)
    # train
    scores = resNet(images)
    loss = criterion(scores, label)
    label_hat = scores.argmax(dim=1)
    # log
    if args.use_tensorboard:
        # compute loss and accuracy
        with torch.no_grad():
            loss_avg_temp += loss.item()
            acc_avg_temp += (label_hat == label).float().mean().item()
    if ((step+1) % args.log_interval == 0):
        # loss average
        loss_avg_temp /= args.log_interval
        acc_avg_temp /= args.log_interval
        writer.add_scalar('test/loss', loss_avg_temp, step+1)
        writer.add_scalar('test/accuracy', acc_avg_temp, step+1)
        loss_avg_temp = 0
        acc_avg_temp = 0

    # store results
    res[step*args.batchSize : (step+1)*args.batchSize] = label_hat.cpu()

# save results to file
resultPath = os.path.join(
    args.res_dir, str(args.checkSession), 'predictions_{}_{}_{}.csv'.format(
    str(args.checkSession), str(args.checkEpoch), str(args.checkPoint)))
if not os.path.exists(os.path.dirname(resultPath)):
    os.makedirs(os.path.dirname(resultPath))

res = res.view(-1, 1)
imageList = torch.arange(0, testSize).view(-1, 1)
res = torch.cat((imageList, res), dim=1)

with open(resultPath, 'wb') as f:
        pickle.dump(res.numpy(), f, pickle.HIGHEST_PROTOCOL)
print(">>> test results saved to {}".format(resultPath))

157it [02:13,  1.18it/s]


>>> test results saved to ./output/predictions/1/predictions_1_10_469.csv
